<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/Fill_gaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql    import SparkSession
from pyspark.sql.window import Window


In [2]:
spark=SparkSession.builder.appName("test").getOrCreate()


In [3]:
df=spark.read.csv("/content/sample_data/transactions_samples_gaps.csv",inferSchema='True',header='True')

In [4]:
df.show()

+----------------+---+-----------+-----------+-------+
|transaction_date| id|customer_id|   custname|    amt|
+----------------+---+-----------+-----------+-------+
|      2026-01-01|  1|       1001|Customer_01|3892.38|
|      2026-01-02|  2|       1001|Customer_01| 2250.5|
|      2026-01-03|  3|       1001|Customer_01|4307.13|
|      2026-01-04|  4|       1001|Customer_01| 3517.1|
|      2026-01-05|  5|       1001|Customer_01| 561.47|
|      2026-01-06|  6|       1001|Customer_01|4880.55|
|      2026-01-07|  7|       1001|Customer_01|3829.58|
|      2026-01-08|  8|       1001|Customer_01|3951.72|
|      2026-01-09|  9|       1001|Customer_01| 727.76|
|      2026-01-10| 10|       1001|Customer_01|2306.89|
|      2026-01-11| 11|       1001|Customer_01|1916.91|
|      2026-01-12| 12|       1001|Customer_01|4641.15|
|      2026-01-13| 13|       1001|Customer_01|3254.94|
|      2026-01-14| 14|       1001|Customer_01|4131.53|
|      2026-01-15| 15|       1001|Customer_01|2272.73|
|      202

In [5]:
df = df.orderBy("customer_id","transaction_date")

In [6]:
window=Window().partitionBy("customer_id").orderBy("transaction_date")


In [7]:
df_lag=df.withColumn("next_transaction_date",lead("transaction_date",1).over(window))

In [8]:
df_lag_gap=df_lag.withColumn("gap",datediff("next_transaction_date","transaction_date"))

In [9]:
df_lag_gap.filter(col("gap")>1).show(100)

+----------------+---+-----------+-----------+-------+---------------------+---+
|transaction_date| id|customer_id|   custname|    amt|next_transaction_date|gap|
+----------------+---+-----------+-----------+-------+---------------------+---+
|      2026-01-06| 75|       1002|Customer_02| 385.68|           2026-01-09|  3|
|      2026-02-14|112|       1002|Customer_02|1091.58|           2026-02-17|  3|
|      2026-01-19|222|       1004|Customer_04|2494.06|           2026-01-21|  2|
|      2026-02-02|235|       1004|Customer_04|4671.81|           2026-02-06|  4|
|      2026-02-28|258|       1004|Customer_04| 615.86|           2026-03-02|  2|
|      2026-01-11|347|       1006|Customer_06|4735.53|           2026-01-15|  4|
|      2026-02-19|383|       1006|Customer_06|3923.48|           2026-02-21|  2|
|      2026-01-24|494|       1008|Customer_08|1649.45|           2026-01-26|  2|
|      2026-02-09|509|       1008|Customer_08|2521.29|           2026-02-13|  4|
|      2026-03-04|529|      

In [10]:
df_empty1=None
df_empty2=None

In [11]:
cnt=df_lag_gap.count()
print(cnt)

667


In [12]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

schema = StructType([
    StructField("transaction_date", DateType(), True),
    StructField("id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("custname", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("next_transaction_date", DateType(), True),
    StructField("gap", IntegerType(), True)
])

df_empty2 = spark.createDataFrame([], schema)

In [13]:
df_lag_gap.printSchema()

root
 |-- transaction_date: date (nullable = true)
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- custname: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- next_transaction_date: date (nullable = true)
 |-- gap: integer (nullable = true)



In [14]:
from datetime import timedelta

In [15]:
from datetime import timedelta
for i in range(cnt):
    #print(df_lag_gap.collect()[i][6])
    gap=df_lag_gap.collect()[i][6]
    if gap is None:
        continue
    j=gap -1
    df_empty1=spark.createDataFrame([], schema)
    for k in range(j):
      print(df_lag_gap.collect()[i][0])
      l_tr_date=df_lag_gap.collect()[i][0] + timedelta(days=k)
      df_rec = spark.createDataFrame([(l_tr_date, df_lag_gap.collect()[i][1], df_lag_gap.collect()[i][2], df_lag_gap.collect()[i][3],df_lag_gap.collect()[i][4],df_lag_gap.collect()[i][5],df_lag_gap.collect()[i][6] )],
                                       ["transaction_date", "id", "customer_id", "custname", "amt","next_transaction_date","gap"])
      df_empty1 = df_empty1.union(df_rec)
    df_empty2=df_empty2.union(df_empty1)
final_df=df_lag_gap.union(df_empty2)



2026-01-06
2026-01-06
2026-02-14
2026-02-14
2026-01-19
2026-02-02
2026-02-02
2026-02-02
2026-02-28
2026-01-11
2026-01-11
2026-01-11
2026-02-19
2026-01-24
2026-02-09
2026-02-09
2026-02-09
2026-03-04
2026-01-02
2026-01-02
2026-01-31
2026-02-24
2026-02-24


In [16]:
cnt1=final_df.count()
print(cnt1)

690


In [18]:
final_df.orderBy("customer_id","transaction_date").show(1000)

+----------------+---+-----------+-----------+-------+---------------------+----+
|transaction_date| id|customer_id|   custname|    amt|next_transaction_date| gap|
+----------------+---+-----------+-----------+-------+---------------------+----+
|      2026-01-01|  1|       1001|Customer_01|3892.38|           2026-01-02|   1|
|      2026-01-02|  2|       1001|Customer_01| 2250.5|           2026-01-03|   1|
|      2026-01-03|  3|       1001|Customer_01|4307.13|           2026-01-04|   1|
|      2026-01-04|  4|       1001|Customer_01| 3517.1|           2026-01-05|   1|
|      2026-01-05|  5|       1001|Customer_01| 561.47|           2026-01-06|   1|
|      2026-01-06|  6|       1001|Customer_01|4880.55|           2026-01-07|   1|
|      2026-01-07|  7|       1001|Customer_01|3829.58|           2026-01-08|   1|
|      2026-01-08|  8|       1001|Customer_01|3951.72|           2026-01-09|   1|
|      2026-01-09|  9|       1001|Customer_01| 727.76|           2026-01-10|   1|
|      2026-01-1